# Period Finding (Shor-style) — Analytic QFT + O(√r) Algorithms

This notebook demonstrates two complementary capabilities:
1. **Analytic QFT sampling** for periodic states (O(1) per shot).
2. **Classical O(√r) period-finding** methods (Pollard's rho, BSGS, etc.).

In [ ]:
from quantum_hybrid_system import QuantumClassicalHybrid, PeriodicState
import numpy as np
from collections import Counter

hybrid = QuantumClassicalHybrid(verbose=False)

# --- Analytic QFT sampling demo ---
n = 10
r = 4
state = PeriodicState(num_qubits=n, offset=0, period=r)

shots = 2000
samples = state.measure(num_shots=shots, use_qft=True)

# QFT of periodic state has peaks at multiples of N/r
N = 2**n
peaks = [k * N // r for k in range(r)]
print("Expected peak centers:", peaks)

# Count how many samples land near each peak (±2 bins)
def near(sample, center, N):
    d = min((sample-center) % N, (center-sample) % N)
    return d <= 2

counts = {p: 0 for p in peaks}
for s in samples:
    for p in peaks:
        if near(s, p, N):
            counts[p] += 1
            break

print("Counts near peaks:", counts)

# --- Classical O(√r) period finding ---
cases = [(7, 15), (2, 91), (3, 221), (5, 437), (7, 899)]
print("\nPeriod-finding results:")
for a, N in cases:
    res = hybrid.find_period(a, N, method="auto")
    print(f"a={a:<2}  N={N:<4}  period={res.period}  method={res.method}  time={res.time_seconds*1e3:.3f} ms")